In [1]:
# Instalações (Google Colab)
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
!pip install selenium beautifulsoup4 > /dev/null
!pip install pandas requests > /dev/null

# Imports
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import os, time, re, requests, zipfile
import pandas as pd
from datetime import datetime

# Setup de pastas
data_hoje = datetime.now().strftime('%Y-%m-%d')
pasta_base = f"{data_hoje}_joli"
pasta_imagens = os.path.join(pasta_base, "imagens")
os.makedirs(pasta_imagens, exist_ok=True)

# Configurar WebDriver
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(options=options)

# COLETAR LINKS
categoria_url = "https://www.joli.com.br/pisos-revestimentos-e-porcelanatos?page="
links = []
pagina = 1

print("🔎 Coletando links dos produtos da Joli (sem limite)...")
while True:
    driver.get(categoria_url + str(pagina))
    time.sleep(3)
    produtos = driver.find_elements(By.CSS_SELECTOR, ".vtex-search-result-3-x-galleryItem a")
    novos_links = [p.get_attribute("href") for p in produtos if p.get_attribute("href") not in links]

    if not novos_links:
        print(f"✅ Fim da coleta na página {pagina}")
        break

    links.extend(novos_links)
    print(f"🔗 Página {pagina}: {len(novos_links)} novos links")
    pagina += 1

driver.quit()

# Função para extrair detalhes
def extrair_detalhes_produto(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        nome = soup.find("h1").get_text(strip=True) if soup.find("h1") else "N/A"

        codigo = soup.find("span", string=lambda t: t and "Referência:" in t)
        codigo = codigo.get_text(strip=True).replace("Referência:", "").strip() if codigo else "0"

        preco_original = soup.select_one(".joli-joli-theme-1-x-list-price .joli-joli-theme-1-x-price")
        preco_original = preco_original.get_text(strip=True) if preco_original else "N/A"

        preco_promocional = soup.select_one(".joli-joli-theme-1-x-selling-price .joli-joli-theme-1-x-price")
        preco_promocional = preco_promocional.get_text(strip=True) if preco_promocional else "N/A"

        preco_m2 = "N/A"

        descricao_tag = soup.find("div", class_="joli-joli-theme-1-x-description")
        descricao_sec = soup.select_one(".joli-joli-theme-1-x-product-description")
        descricao = descricao_tag.get_text(strip=True) if descricao_tag else (
            descricao_sec.get_text(separator="\n", strip=True) if descricao_sec else "Descrição não encontrada"
        )

        especificacoes = {}
        ficha_tecnica_tag = soup.find("div", class_="joli-joli-theme-1-x-specifications")
        if ficha_tecnica_tag:
            linhas = ficha_tecnica_tag.find_all("div", class_="joli-joli-theme-1-x-specification")
            for linha in linhas:
                chave = linha.find("span", class_="joli-joli-theme-1-x-specificationName")
                valor = linha.find("span", class_="joli-joli-theme-1-x-specificationValue")
                if chave and valor:
                    especificacoes[chave.get_text(strip=True)] = valor.get_text(strip=True)
        especificacoes_str = " | ".join([f"{k}: {v}" for k, v in especificacoes.items()]) if especificacoes else "N/A"

        img_tag = soup.select_one("img.joli-store-components-0-x-productImageTag--main")
        img_url = img_tag["src"] if img_tag else "N/A"

        return [nome, codigo, especificacoes_str, descricao, preco_original, preco_promocional, preco_m2, img_url, url]
    except Exception as e:
        print(f"❌ Erro em {url}: {e}")
        return None

# Função para salvar imagem
def salvar_imagem(url_img, nome_base):
    if url_img == "N/A": return
    try:
        resposta = requests.get(url_img, timeout=10)
        if resposta.status_code == 200:
            caminho = os.path.join(pasta_imagens, f"{nome_base}.jpg")
            with open(caminho, 'wb') as f:
                f.write(resposta.content)
    except Exception as e:
        print(f"❌ Erro ao baixar imagem: {e}")

# COLETAR DADOS
dados_produtos = []
for i, link in enumerate(links):
    print(f"[{i+1}/{len(links)}] Coletando dados de: {link}")
    resultado = extrair_detalhes_produto(link)
    if resultado:
        dados_produtos.append(resultado)
        codigo = re.sub(r'[\\/*?:"<>|]', "", resultado[1])
        if not codigo or codigo == "0":
            codigo = re.sub(r'[\\/*?:"<>|]', "", resultado[0])[:50]
        salvar_imagem(resultado[7], codigo)

# Criar DataFrame
df = pd.DataFrame(dados_produtos, columns=[
    "Nome", "Código", "Especificações", "Descrição", "Preço Original",
    "Preço Promocional", "Preço por m²", "Imagem", "URL"
])

# Limpar preços
def limpar_preco(valor):
    try:
        return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        return None

df["Preço Original"] = df["Preço Original"].apply(limpar_preco)
df["Preço Promocional"] = df["Preço Promocional"].apply(limpar_preco)
df["Preço por m²"] = df["Preço por m²"].apply(limpar_preco)

# Adicionar coluna da loja
df["Loja"] = "Joli"

# Reorganizar colunas
df = df[[
    "Código", "Nome", "Preço Promocional", "Preço Original", "Preço por m²",
    "Especificações", "Descrição", "Imagem", "URL", "Loja"
]]

# Salvar CSV
csv_path = os.path.join(pasta_base, f"joli_{data_hoje}.csv")
df.to_csv(csv_path, sep='|', index=False, encoding='utf-8-sig')
print(f"\n📄 CSV organizado salvo em: {csv_path}")

# Compactar imagens
zip_path = os.path.join(pasta_base, f"imagens_joli_{data_hoje}.zip")
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for root, dirs, files in os.walk(pasta_imagens):
        for file in files:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"📦 Imagens compactadas em: {zip_path}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
🔎 Coletando links dos produtos da Joli (sem limite)...
🔗 Página 1: 6 novos links
🔗 Página 2: 6 novos links
🔗 Página 3: 6 novos links
🔗 Página 4: 6 novos links
🔗 Página 5: 6 novos links
🔗 Página 6: 6 novos links
🔗 Página 7: 6 novos links
🔗 Página 8: 6 novos links
🔗 Página 9: 6 novos links
🔗 Página 10: 6 novos links
🔗 Página 11: 6 novos links
🔗 Página 12: 6 novos links
🔗 Página 13: 6 novos links
🔗 Página 14: 6 novos links
🔗 Página 15: 6 novos links
🔗 Página 16: 6 novos links
🔗 Página 17: 6 novos links
🔗 Página 18: 6 novos links
🔗 Página 19: 6 novos links
🔗 Página 20: 6 novos links
🔗 Página 21: 6 novos links
🔗 Página 22: 6 novos links
🔗 Página 23: 6 novos links
🔗 Página 24: 6 novos links
🔗 Página 25: 6 novos lin